# 03 — Patch Extraction and Normalisation

**Purpose:** Convert masked full-sky HEALPix maps into the flat-sky training patches
consumed by `train.py`.

This notebook completes the preprocessing pipeline in three steps:

1. **Low-pass filtering** — applies a sharp spherical-harmonic cutoff at ℓ = 7000 to
   remove aliasing artifacts, then zeros any negative pixels introduced by the filter
   (Gibbs ringing around masked regions).

2. **Flat-sky patch extraction** — uses `FlatCutter` to project 6°×6° patches onto
   256×256 Cartesian grids (1.41 arcmin/pixel) at centres computed by `get_patch_centers`
   with a 6° step size.

3. **Normalisation and saving** — min-max normalises all patches jointly to [0, 1] and
   saves them as `(N, 256, 256, 1)` arrays. Also generates correlated Gaussian
   realisations from the measured CIB–tSZ cross-power spectrum for use as a
   non-Gaussian baseline in validation.

**Inputs:**
- Masked CIB map: `data/cib_150_masked.fits` (from `02_masking.ipynb`)
- Masked tSZ map: `data/tsz_150_masked.fits` (from `02_masking.ipynb`)

**Outputs:**
- CIB patches: `data/low_pass/2mJy/CIB_map_150GHz_256_st6_minmax_2mJy_zero_lp.npy`
- tSZ patches: `data/low_pass/2mJy/tSZ3_map_150GHz_256_st6_minmax_2mJy_norm_lp.npy`
- Gaussian baseline patches: `data/low_pass/2mJy/gaussian_cib_tsz_2mJy_lp.npy`

**Key module functions:**
- `foregrounds_diffusion.preprocessing.get_patch_centers`
- `foregrounds_diffusion.preprocessing.FlatCutter`
- `foregrounds_diffusion.preprocessing.apply_maxmin_normalization`
- `foregrounds_diffusion.flatmaps.make_gaussian_realisation`

**Paper reference:** §2 (Figure 1 pipeline), §3.2 (Gaussian baseline construction).

In [ ]:
import numpy as np
import healpy as hp
import astropy.units as u
from pathlib import Path

from foregrounds_diffusion.preprocessing import (
    get_patch_centers, FlatCutter, apply_maxmin_normalization, apply_stdnorm,
)
from foregrounds_diffusion.flatmaps import map2cl, make_gaussian_realisation

PATCH_DEG  = 6.0   # patch side length in degrees
NRES       = 256   # pixels per side
STEP_DEG   = 6.0   # patch centre spacing in degrees
GAL_CUT    = 20.0  # Galactic-plane exclusion half-width in degrees
LMAX_FILTER = 7000  # low-pass cut-off multipole
PTSRC      = 2     # point-source threshold label (mJy)

flatskymapparams = [NRES, NRES, PATCH_DEG * 60. / NRES, PATCH_DEG * 60. / NRES]
# => [256, 256, 1.40625, 1.40625] arcmin/pixel
print(f"Pixel resolution: {flatskymapparams[2]:.4f} arcmin")


In [ ]:
cib_map = hp.read_map("data/cib_150_masked.fits")
tsz_map = hp.read_map("data/tsz_150_masked.fits")
nside   = hp.get_nside(cib_map)

# Low-pass filter at ell = LMAX_FILTER via spherical harmonic transform
print(f"Applying low-pass filter at ell = {LMAX_FILTER} …")
cib_alm = hp.map2alm(cib_map, lmax=LMAX_FILTER)
tsz_alm = hp.map2alm(tsz_map, lmax=LMAX_FILTER)
cib_lp  = hp.alm2map(cib_alm, nside=nside)
tsz_lp  = hp.alm2map(tsz_alm, nside=nside)

# Zero negative pixels (Gibbs ringing artefacts around masked regions)
n_neg_cib = (cib_lp < 0).sum()
print(f"Zeroing {n_neg_cib:,} negative CIB pixels ({100 * n_neg_cib / len(cib_lp):.2f}%)")
cib_lp[cib_lp < 0] = 0.


In [ ]:
centers = get_patch_centers(gal_cut=GAL_CUT * u.deg, step_size=STEP_DEG * u.deg)
print(f"Patch centres (|b| > {GAL_CUT}°, step {STEP_DEG}°): {len(centers)}")

cutter = FlatCutter(ang_x=PATCH_DEG * u.deg, ang_y=PATCH_DEG * u.deg,
                    xres=NRES, yres=NRES)

cib_patches, tsz_patches = [], []
for lon, lat in centers:
    patch = cutter.rotate_to_pole_and_interpolate(lon, lat, [cib_lp, tsz_lp])
    cib_patches.append(patch[:, :, 0:1])
    tsz_patches.append(patch[:, :, 1:2])

cib_patches = np.stack(cib_patches)  # (N, 256, 256, 1)
tsz_patches = np.stack(tsz_patches)  # (N, 256, 256, 1)
print(f"Extracted {len(cib_patches)} patches, shape {cib_patches.shape}")


In [ ]:
# CIB: global min-max → [0, 1]
cib_norm = apply_maxmin_normalization(cib_patches)

# tSZ: per-channel standard normalisation (zero mean, unit std)
tsz_norm = apply_stdnorm(tsz_patches)

print(f"CIB — min: {cib_norm.min():.4f}  max: {cib_norm.max():.4f}")
print(f"tSZ — mean: {tsz_norm.mean():.4f}  std: {tsz_norm.std():.4f}")


In [ ]:
# Measure mean auto- and cross-power spectra from normalised patches
LMIN, LMAX, LBIN = 300, 7000, 100
N_FOR_SPECTRA = min(200, len(cib_norm))

el_arr, cl_cib_arr, cl_tsz_arr, cl_cross_arr = [], [], [], []
for i in range(N_FOR_SPECTRA):
    cib_m = cib_norm[i, :, :, 0]
    tsz_m = tsz_norm[i, :, :, 0]
    el, cl_c  = map2cl(flatskymapparams, cib_m, binsize=LBIN, minbin=LMIN, maxbin=LMAX)
    _,  cl_t  = map2cl(flatskymapparams, tsz_m, binsize=LBIN, minbin=LMIN, maxbin=LMAX)
    _,  cl_x  = map2cl(flatskymapparams, cib_m, tsz_m, binsize=LBIN, minbin=LMIN, maxbin=LMAX)
    cl_cib_arr.append(cl_c);  cl_tsz_arr.append(cl_t);  cl_cross_arr.append(cl_x)
el_arr = el

cl_cib_mean  = np.mean(cl_cib_arr,  axis=0)
cl_tsz_mean  = np.mean(cl_tsz_arr,  axis=0)
cl_cross_mean = np.mean(cl_cross_arr, axis=0)
print(f"Mean CIB  C_ell range: {cl_cib_mean.min():.3e} – {cl_cib_mean.max():.3e}")
print(f"Mean tSZ  C_ell range: {cl_tsz_mean.min():.3e} – {cl_tsz_mean.max():.3e}")

# Generate correlated Gaussian realisations from the measured spectra
N_GAUSSIAN = len(cib_norm)
gaussian_maps = []
for _ in range(N_GAUSSIAN):
    sim = make_gaussian_realisation(
        flatskymapparams, el_arr,
        cl=cl_cib_mean, cl2=cl_tsz_mean, cl12=cl_cross_mean,
    )
    gaussian_maps.append(sim[:2])  # keep only the two correlated fields

gaussian_arr = np.array(gaussian_maps)  # (N, 2, H, W)  channels-first
print(f"Gaussian baseline shape: {gaussian_arr.shape}")


In [ ]:
OUT_DIR = Path(f"data/low_pass/{PTSRC}mJy")
OUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(OUT_DIR / f"CIB_map_150GHz_{NRES}_st6_minmax_{PTSRC}mJy_zero_lp.npy",  cib_norm)
np.save(OUT_DIR / f"tSZ3_map_150GHz_{NRES}_st6_minmax_{PTSRC}mJy_norm_lp.npy", tsz_norm)
np.save(OUT_DIR / f"gaussian_cib_tsz_{PTSRC}mJy_lp.npy", gaussian_arr)
print("Saved all patch arrays.")
